# Agentic Multi-Hop RAG with Sub-Query Decomposition

## 1. Beyond Single-Hop Vector Retrieval
Single-hop vector retrieval assumes that a user's prompt can be answered by a single semantic neighborhood. However, complex real-world queries frequently require **multi-hop reasoning**:

> *"How did NVIDIA's data center revenue growth in Q3 2024 compare with AMD's data center segment over the same quarter, and what were the main drivers?"*

A naive vector search on this entire prompt fails because no single document discusses both company earnings reports in unified depth. The query collapses into semantic noise.

### Agentic RAG Architecture
Agentic RAG treats retrieval as a **tool-calling decision process**:
1. **Decomposer**: Breaks down the complex multi-part query into atomic sub-questions.
2. **Router**: Dynamically dispatches each sub-question to specialized data stores (SQL database, vector search, web search, or structured API).
3. **Iterative Evaluator**: Evaluates whether retrieved evidence answers the sub-question. If incomplete, generates a refined follow-up query.
4. **Synthesizer**: Fuses intermediate answers into a structured final response.

In [ ]:
import json

class AgenticRAGCoordinator:
    """
    Demonstrates autonomous multi-hop query decomposition, routing, and synthesis.
    """
    def __init__(self):
        # Simulated tools and data silos
        self.sql_db = {
            "nvidia_q3_2024_datacenter": "USD 30.8 Billion (+112% YoY)",
            "amd_q3_2024_datacenter": "USD 3.5 Billion (+122% YoY)"
        }
        self.vector_store = {
            "nvidia_growth_drivers": "Hopper H100/H200 architecture and Blackwell sampling driven by cloud hyperscalers.",
            "amd_growth_drivers": "Instinct MI300X GPU accelerator ramp-up with enterprise AI customers."
        }

    def decompose_query(self, complex_query):
        """Breaks complex comparison query into discrete sub-tasks."""
        return [
            {"id": "step_1", "task": "Lookup NVIDIA and AMD Q3 2024 data center revenues", "target": "sql_engine"},
            {"id": "step_2", "task": "Retrieve NVIDIA revenue growth drivers", "target": "vector_rag"},
            {"id": "step_3", "task": "Retrieve AMD revenue growth drivers", "target": "vector_rag"}
        ]

    def route_and_execute(self, task_spec):
        """Dispatches sub-task to corresponding data source."""
        target = task_spec["target"]
        t_id = task_spec["id"]
        
        if target == "sql_engine":
            nv_rev = self.sql_db["nvidia_q3_2024_datacenter"]
            amd_rev = self.sql_db["amd_q3_2024_datacenter"]
            return f"NVIDIA: {nv_rev}; AMD: {amd_rev}"
        elif target == "vector_rag":
            if "nvidia" in task_spec["task"].lower():
                return self.vector_store["nvidia_growth_drivers"]
            else:
                return self.vector_store["amd_growth_drivers"]
        return "Not found"

    def synthesize(self, complex_query, sub_results):
        """Synthesizes final cohesive response from intermediate evidence."""
        rev_info = sub_results["step_1"]
        nv_drivers = sub_results["step_2"]
        amd_drivers = sub_results["step_3"]
        
        report = (
            f"### Comprehensive Financial Comparison Analysis\n"
            f"- **Revenue Findings**: {rev_info}\n"
            f"- **NVIDIA Drivers**: {nv_drivers}\n"
            f"- **AMD Drivers**: {amd_drivers}\n"
            f"- **Synthesis**: Both companies showed explosive triple-digit data center growth in Q3 2024, "
            f"with NVIDIA leading absolute scale (30.8B) via Hopper/Blackwell, and AMD exhibiting rapid catch-up "
            f"(122% YoY) powered by the MI300X series."
        )
        return report

# Execution Run
coordinator = AgenticRAGCoordinator()
user_query = "Compare NVIDIA vs AMD Q3 2024 data center revenue and explain their growth drivers."

sub_tasks = coordinator.decompose_query(user_query)
print(f"User Query: {user_query}")
print(f"Decomposed into {len(sub_tasks)} distinct sub-tasks:
")

intermediate_findings = {}
for task in sub_tasks:
    ans = coordinator.route_and_execute(task)
    intermediate_findings[task["id"]] = ans
    print(f"  [{task['id']}] Target: {task['target']:12s} -> {ans}")

final_answer = coordinator.synthesize(user_query, intermediate_findings)
print("
" + final_answer.replace('\n', '
'))

## 2. Dynamic Routing Matrix: Tool Selection Criteria

| Tool / Engine | Query Characteristic | Latency Profile | Example Use Case |
| :--- | :--- | :--- | :--- |
| **SQL Database Engine** | Exact metrics, aggregates, dates, tabular math | <10ms | "Total revenue in Q3 by region" |
| **Vector DB (Bi-Encoder)** | Unstructured qualitative semantic concepts | 15-50ms | "Customer sentiment on battery life" |
| **Web Search API (Live)** | Real-time events post-training cutoff | 300-1000ms | "Stock price right now or recent breaking news" |
| **Graph Database (KG)** | Multi-entity relational link tracing | 20-80ms | "Find all subsidiaries owned by parent corp" |

## 3. High-Yield Interview Questions & Edge Cases

### Q1: How do you prevent infinite execution loops in autonomous Agentic RAG systems?
**Answer:**
1. **Max Iteration Budget**: Hardcap max reflection/retrieval steps (e.g. $N_{\max} = 4$).
2. **Deterministic DAG Planning**: Prefer pre-planned Directed Acyclic Graphs over unconstrained ReAct loops where sub-tasks have explicit dependencies.
3. **Similarity Convergence Checks**: If consecutive queries retrieve near-identical documents (cosine similarity $>0.92$), abort retrieval and force generation with currently available evidence.

### Q2: What is the difference between Self-RAG and Corrective RAG (CRAG)?
**Answer:**
- **Self-RAG**: Uses special reflection tokens (`[Retrieve]`, `[IsRel]`, `[IsSup]`) trained into the LLM itself to self-judge whether retrieval is needed and whether generated text is supported by facts.
- **Corrective RAG (CRAG)**: Uses an external evaluator model to assess retrieval quality into three discrete confidence states: *Correct* (proceed to generation), *Incorrect* (discard documents and trigger web search fallback), or *Ambiguous* (combine vector search and web search).